In [2]:
from heart_disease.data.ingestion import load_csv
from heart_disease.features.preprocessing import create_training_pipeline, create_categorical_pipeline, create_numeric_pipeline, create_preprocessor
import heart_disease.models.train as t
from heart_disease.config import INTERIM_DATA_DIR, RANDOM_STATE, TARGET_COLUMN, FEATURES, MODEL_DIR
from heart_disease.features.cleaning import clean_data, calculate_missing_data
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
import pandas as pd

# Import and Split Data

In [3]:
df = load_csv(INTERIM_DATA_DIR / "cleaned_dataset.csv")
df.head()

,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,target
0,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,1
2,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [8]:
X_train, X_test, y_train, y_test = t.split_data(df, TARGET_COLUMN, FEATURES)
X_test.to_parquet(INTERIM_DATA_DIR / 'X_test.parquet', index=False)
y_test.to_frame("target").to_parquet(INTERIM_DATA_DIR / 'y_test.parquet', index=False)

2026-08-31 01:15:57,496 | INFO | heart_disease.models.train | Created with shape of X_train: (736, 14), X_test: (184, 14), y_train: (736,), y_test: (184,)


# Build Baseline Model

In [4]:
logistic_regression = create_training_pipeline(LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))
t.cross_validate_model(logistic_regression, X_train, y_train, t.TrainingConfig())

2026-08-30 23:08:06,342 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-08-30 23:08:08,525 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.832 ± 0.011


,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
0,0.816568,0.823529,0.835443,0.836364,0.849673,0.832315,0.011425


# Evaluate Dataset Based On Model Performance

## Evaluate maximum missing data allowed in each observation

Each observation contains fourteen features that may provide useful predictive information.
However, some observation have more than half of their features missing, potentially indicating poor data quality. 
Therefore, we evaluate different threshold for removing observations based on their proportion of missing values using cross-validation. 
The selected threshold must satisfy two constraint: the proportion of discarded observations must not exceed `ten percent` of the dataset, and the refined dataset must have a similar target mean to the original dataset.

In [5]:
results = []

for num in range(7, 15):
    initial_rows = X_train.shape[0]
    temporary_df = X_train.copy()
    temporary_df[TARGET_COLUMN] = y_train.copy()
    temporary_df = temporary_df.dropna(thresh=num)

    X = temporary_df[FEATURES]
    y = temporary_df[TARGET_COLUMN]

    scores = t.cross_validate_model(
        logistic_regression, 
        X, 
        y, 
        t.TrainingConfig()
    )
    
    scores['threshold'] = num
    scores['total_rows'] = X.shape[0]
    scores['%_rows_deleted'] = (initial_rows - X.shape[0]) / initial_rows * 100
    scores["total_rows x mean"] = scores["total_rows"] * scores["mean"]
    scores['target_mean'] = y.mean()
    results.append(scores)

results = pd.concat(results)
results = results.sort_values("total_rows x mean", ascending=False).set_index("threshold")
results

2026-08-30 23:08:08,636 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1


2026-08-30 23:08:09,760 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.832 ± 0.011
2026-08-30 23:08:09,775 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-08-30 23:08:10,825 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.844 ± 0.019
2026-08-30 23:08:10,839 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-08-30 23:08:11,753 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.852 ± 0.034
2026-08-30 23:08:11,774 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-08-30 23:08:12,700 | IN

,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std,total_rows,%_rows_deleted,total_rows x mean,target_mean
threshold,,,,,,,,,,,
8,0.817073,0.857143,0.825000,0.853503,0.867470,0.844038,0.019493,727,1.222826,613.615494,0.551582
7,0.816568,0.823529,0.835443,0.836364,0.849673,0.832315,0.011425,736,0.000000,612.584184,0.552989
9,0.800000,0.872483,0.887417,0.825000,0.877419,0.852464,0.033918,690,6.250000,588.200132,0.546377
10,0.851351,0.858974,0.810811,0.861111,0.883117,0.853073,0.023627,688,6.521739,586.914157,0.545058
11,0.870130,0.829630,0.828571,0.861314,0.819444,0.841818,0.020031,649,11.820652,546.339784,0.546995
12,0.850000,0.877193,0.904348,0.873950,0.833333,0.867765,0.024358,474,35.597826,411.320489,0.613924
13,0.869565,0.857143,0.868687,0.888889,0.844444,0.865746,0.014742,401,45.516304,347.164008,0.586035
14,0.808511,0.800000,0.840000,0.830189,0.875000,0.830740,0.026397,262,64.402174,217.653844,0.473282


Based on the result of testing multiple threshold above 

In [6]:
mask = X_train.notna().sum(axis=1) >= 8

X_train = X_train.loc[mask]
y_train = y_train.loc[mask]


In [7]:
strategies = {
    "indicators_both": create_preprocessor(
        numeric_pipeline=create_numeric_pipeline(add_indicator=True),
        categorical_pipeline=create_categorical_pipeline(add_indicator=True),
    ),
    "indicators_numeric": create_preprocessor(
        numeric_pipeline=create_numeric_pipeline(add_indicator=False),
        categorical_pipeline=create_categorical_pipeline(add_indicator=True),
    ),
    "indicators_categorical": create_preprocessor(
        numeric_pipeline=create_numeric_pipeline(add_indicator=False),
        categorical_pipeline=create_categorical_pipeline(add_indicator=False),
    ),
    "indicators_none": create_preprocessor(
        numeric_pipeline=create_numeric_pipeline(add_indicator=False),
        categorical_pipeline=create_categorical_pipeline(add_indicator=False),
    ),
}

In [8]:
list_of_scores = []

for name, strategy in strategies.items():
    pipeline = create_training_pipeline(
        model=LogisticRegression(random_state=RANDOM_STATE),
        preprocessor=strategy
    )

    scores = t.cross_validate_model(
        pipeline, 
        X_train,
        y_train,
        t.TrainingConfig()
    )

    scores['strategy'] = name
    list_of_scores.append(scores)

list_of_scores = pd.concat(list_of_scores).sort_values("mean", ascending=False).set_index("strategy")
list_of_scores

2026-08-30 23:08:16,043 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-08-30 23:08:17,133 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.852 ± 0.034
2026-08-30 23:08:17,136 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-08-30 23:08:18,290 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.853 ± 0.031
2026-08-30 23:08:18,297 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-08-30 23:08:18,945 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.827 ± 0.025
2026-08-30 23:08:18,955 | IN

,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
strategy,,,,,,,
indicators_numeric,0.800000,0.870748,0.881579,0.835443,0.877419,0.853038,0.031130
indicators_both,0.800000,0.872483,0.887417,0.825000,0.877419,0.852464,0.033918
indicators_categorical,0.794521,0.813333,0.862745,0.812500,0.849673,0.826554,0.025467
indicators_none,0.794521,0.813333,0.862745,0.812500,0.849673,0.826554,0.025467


# Evaluate Different Algorithm 

In [9]:
list_of_models = [
    LogisticRegression(random_state=42),
    RandomForestClassifier(random_state=42),
    CalibratedClassifierCV(SVC(), ensemble=False),
    GradientBoostingClassifier(random_state=42),
    KNeighborsClassifier(),
    XGBClassifier(random_state=42)
]

In [10]:
t.model_comparison_cv(list_of_models, create_training_pipeline, X_train, y_train, t.TrainingConfig())

,Model,fit_time,Accuracy,Precision,Recall,F1,ROC AUC
0,CalibratedClassifierCV,0.1779,0.8406,0.8424,0.8730,0.8566,0.8978
1,LogisticRegression,0.1226,0.8377,0.8478,0.8597,0.8525,0.9090
2,RandomForestClassifier,0.4285,0.8333,0.8410,0.8571,0.8484,0.8832
3,XGBClassifier,0.2325,0.8188,0.8283,0.8438,0.8355,0.8687
4,GradientBoostingClassifier,0.4078,0.8188,0.8345,0.8358,0.8342,0.8813
5,KNeighborsClassifier,0.0358,0.8058,0.8154,0.8359,0.8246,0.8654


# Tune Hyperparameters for Selected Algorithm

In [11]:
param_grid = [
    {
        "classifier__max_iter": [5000],
        "classifier__solver": ["lbfgs"],
        "classifier__C": [0.01, 0.1, 1, 10, 100],
    },
    {
        "classifier__max_iter": [5000],
        "classifier__solver": ["saga"],
        "classifier__C": [0.01, 0.1, 1, 10, 100],
        "classifier__l1_ratio": [0, 0.5, 1],
    },
]

In [12]:
search = t.grid_search(
    logistic_regression,
    param_grid,
    X_train,
    y_train,
    t.TrainingConfig()
)

2026-08-30 23:08:29,402 | INFO | heart_disease.models.train | Starting grid search
2026-08-30 23:09:05,620 | INFO | heart_disease.models.train | Best parameters: {'classifier__C': 1, 'classifier__l1_ratio': 0.5, 'classifier__max_iter': 5000, 'classifier__solver': 'saga'} | Best CV scores: 0.855


In [13]:
search.best_score_

np.float64(0.8545525665665898)

In [14]:
best_model = create_training_pipeline(LogisticRegression(max_iter= 5000, l1_ratio=0.5, C=1, solver='saga', random_state=RANDOM_STATE))
t.cross_validate_model(best_model, X_train, y_train, t.TrainingConfig())

2026-08-30 23:09:05,688 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-08-30 23:09:09,776 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.855 ± 0.032


,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
0,0.8,0.872483,0.887417,0.835443,0.877419,0.854553,0.032452


In [15]:
logistic_regression = create_training_pipeline(LogisticRegression(random_state=RANDOM_STATE))
t.cross_validate_model(logistic_regression, X_train, y_train, t.TrainingConfig())

2026-08-30 23:09:09,836 | INFO | heart_disease.models.train | Starting cross validation for LogisticRegression model with cv: StratifiedKFold(n_splits=5, random_state=42, shuffle=True) and scoring: f1
2026-08-30 23:09:11,060 | INFO | heart_disease.models.train | Finishing cross validation with scores: CV F1: 0.852 ± 0.034


,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
0,0.8,0.872483,0.887417,0.825,0.877419,0.852464,0.033918


In [16]:
logistic_regression = t.train_model(best_model, X_train, y_train)

2026-08-30 23:09:12,490 | INFO | heart_disease.models.train | Training: LogisticRegression with shape of dataframe: (690, 14)


In [18]:
path = MODEL_DIR/"logistic_regression.joblib"
t.save_model(logistic_regression, path)

2026-08-30 23:09:40,200 | INFO | heart_disease.models.train | Saving LogisticRegression model to /home/irasionize/heart-disease-ml/models/logistic_regression.joblib
